# RAG with Local LLM and Embeddings

This notebook demonstrates the implementation of a Retrieval-Augmented Generation (RAG) system using:
- **Local LLM Model**: Qwen2.5 from Hugging Face
- **Local Embedding Model**: all-MiniLM-L6-v2 from Hugging Face
- **Multiple Document Types**: PDF, Text, and HTML files
- **Vector Store**: ChromaDB for efficient document retrieval

## Business Application and Context

Imagine having the capability to directly ask questions about Python programming from multiple sources - official documentation, programming books, and pandas library guides - and get immediate, relevant answers. This ability to extract and organize specific insights quickly and accurately enables you to focus on higher-level analysis and decision-making, rather than being bogged down by information retrieval.

In this exercise, you're using a **RAG** model to answer questions from multiple document types, demonstrating how this approach can be scaled to handle complex document repositories in real-world industry contexts.

## Scaling RAG to Multiple Document Types

- **Processing Multiple File Formats:** This notebook processes PDF files, plain text files, and HTML files, demonstrating how RAG systems can handle diverse document formats.
- **Scaling to Multiple Documents:** By processing 10+ text files and 10+ HTML files along with PDF documents, we demonstrate how the system can be scaled to handle larger document repositories.
- **Enhanced Knowledge Retrieval:** A scaled-up RAG system can help developers quickly find relevant information across documentation, tutorials, and reference materials.
- **Personalized Learning Assistant:** Scaled-up RAG systems can transform learning by quickly retrieving relevant information from multiple sources, improving comprehension and productivity.

## Using Local Models

### Why Use Local Models?

**Cost Efficiency:**
- No API costs for embeddings or LLM inference
- Especially beneficial for high-volume applications

**Privacy and Security:**
- Data remains on your infrastructure
- No external API calls with sensitive information
- Compliance with data governance policies

**Customization:**
- Fine-tune models for specific domains
- Full control over model behavior

**Offline Capability:**
- Works without internet connectivity
- No dependency on external service availability

## 0 - Prerequisites and Installation

Before running this notebook, ensure you have sufficient system resources:
- **RAM**: At least 8GB (16GB recommended for larger models)
- **Storage**: At least 10GB free space for models
- **GPU**: Optional but recommended for faster inference (CUDA-compatible GPU)

Install the required packages below.

# Install required packages
## Install PyTorch with CUDA support (if you have a compatible GPU)

```shell
pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126
```

## Install other dependencies
```shell
pip install -q transformers \
              sentence-transformers \
              langchain \
              langchain-community \
              langchain-text-splitters \
              langchain-huggingface \
              chromadb \
              pypdf \
              beautifulsoup4 \
              lxml \
              accelerate \
              bitsandbytes \
              unstructured
```


**Note:** After installation, you may need to restart the runtime. If you encounter any import errors, restart the kernel and continue from the next cell.

## 1 - Import Required Libraries

Import all necessary libraries for document processing, embeddings, vector storage, and LLM interaction.

In [17]:
# Standard library imports
import json
import os
import warnings
import glob
from pathlib import Path

# Data processing imports
import numpy as np
import pandas as pd

# LangChain imports for document loading and processing
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    UnstructuredHTMLLoader,
    DirectoryLoader
)
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Hugging Face Transformers imports for local LLM
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig
)
import torch

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


## 2 - Load Local LLM Model (Qwen2.5)

We will load the Qwen2.5 model from Hugging Face. This is a powerful open-source language model that can run locally.

**About Qwen2.5:**
- Developed by Alibaba Cloud
- Strong performance on various NLP tasks
- Supports multiple languages
- Can run efficiently on consumer hardware

**Memory Optimization:**
We'll use 4-bit quantization to reduce memory requirements while maintaining good performance.

In [18]:
# Configure model quantization for efficient memory usage
# This reduces model size from ~14GB to ~4GB with minimal performance loss
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Model name - using a smaller variant for better performance
# You can change this to "Qwen/Qwen2.5-7B-Instruct" for a larger model
model_name = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading model: {model_name}")
print("This may take a few minutes on first run as the model is downloaded...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

print("✓ Model and tokenizer loaded successfully!")
print(f"Model memory footprint: ~{model.get_memory_footprint() / 1e9:.2f} GB")

Loading model: Qwen/Qwen2.5-3B-Instruct
This may take a few minutes on first run as the model is downloaded...


Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.73s/it]


✓ Model and tokenizer loaded successfully!
Model memory footprint: ~2.01 GB


### Create Text Generation Pipeline

Create a pipeline for easy text generation with the loaded model.

In [19]:
# Create a text generation pipeline
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.2,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print("✓ Text generation pipeline created successfully!")

# Test the pipeline with a simple query
test_response = llm_pipeline("Hello! What is Python?", max_new_tokens=50)
print("\nTest generation:")
print(test_response[0]['generated_text'][:200] + "...")

Device set to use cuda:0


✓ Text generation pipeline created successfully!

Test generation:
Hello! What is Python? Can you explain it to me in simple terms?
Sure, I'd be happy to help!

Python is a programming language that was created by Guido van Rossum and first released in 1991. It's kno...


## 3 - Load Local Embedding Model (all-MiniLM-L6-v2)

We will load the all-MiniLM-L6-v2 embedding model from Hugging Face. This model converts text into vector representations for similarity search.

**About all-MiniLM-L6-v2:**
- Compact and efficient (only 80MB)
- 384-dimensional embeddings
- Excellent performance for semantic similarity tasks
- Fast inference speed
- Well-suited for RAG applications

In [20]:
# Load the embedding model from Hugging Face
# This model will convert text into 384-dimensional vectors
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {embedding_model_name}")

embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={'device': 'cpu'},  # Use 'cuda' if you have a GPU
    encode_kwargs={'normalize_embeddings': True}  # Normalize for cosine similarity
)

print("✓ Embedding model loaded successfully!")

# Test the embedding model
test_text = "This is a test sentence."
test_embedding = embedding_model.embed_query(test_text)
print(f"Embedding dimension: {len(test_embedding)}")
print(f"Sample embedding values: {test_embedding[:5]}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
✓ Embedding model loaded successfully!
Embedding dimension: 384
Sample embedding values: [0.08429647982120514, 0.057953767478466034, 0.00449337437748909, 0.10582111775875092, 0.007083370350301266]


## 4 - Understanding RAG Architecture

### What is RAG?

**Retrieval-Augmented Generation (RAG)** is a powerful technique that combines:
1. **Information Retrieval**: Finding relevant documents from a knowledge base
2. **Text Generation**: Using an LLM to generate responses based on retrieved documents

### RAG Workflow

The RAG process consists of two main phases:

**Phase 1: Indexing (Offline)**
1. Load documents from various sources
2. Split documents into smaller chunks
3. Generate embeddings for each chunk
4. Store embeddings in a vector database

**Phase 2: Querying (Online)**
1. User submits a question
2. Question is converted to an embedding
3. Similar document chunks are retrieved from vector database
4. Retrieved chunks are added as context to the prompt
5. LLM generates answer based on context

### Why Chunking is Important

Chunking solves two critical problems:
1. **Context Window Limitations**: LLMs have token limits (~4K-32K tokens)
2. **Semantic Granularity**: Smaller chunks provide more precise retrieval

**Key Chunking Parameters:**
- **chunk_size**: Number of characters per chunk (we use 512)
- **chunk_overlap**: Overlap between chunks to maintain context continuity (we use 50)

## 5 - Load and Chunk PDF File

We will load the Python Programming PDF file, split it into chunks, and prepare it for embedding.

In [21]:
import os

current_directory = os.getcwd()
print("Current working directory:", current_directory)

Current working directory: c:\Users\I838159\OneDrive - SAP SE\Documents\Files\Personal\AgenticAICert\Samples\01 - Simple Rag


In [22]:
# Path to the PDF file
pdf_file = "../UnstructureData/PDFFiles/Python Programming.pdf"

print(f"Loading PDF file: {pdf_file}")

# Load the PDF
pdf_loader = PyPDFLoader(pdf_file)

# Create a text splitter for chunking
# Using RecursiveCharacterTextSplitter which intelligently splits on paragraph/sentence boundaries
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,      # Maximum characters per chunk
    chunk_overlap=50,    # Overlap between chunks for context continuity
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  # Split hierarchy: paragraphs > lines > words > chars
)

# Load and split the PDF
pdf_chunks = pdf_loader.load_and_split(text_splitter)

print(f"✓ PDF loaded and split into {len(pdf_chunks)} chunks")
print(f"\nSample chunk (first 300 characters):")
print(pdf_chunks[0].page_content[:300] + "...")

Loading PDF file: ../UnstructureData/PDFFiles/Python Programming.pdf
✓ PDF loaded and split into 328 chunks

Sample chunk (first 300 characters):
Python ProgrammingHans-Petter Halvorsen
https://www.halvorsen.blog...


## 6 - Load and Chunk Text Files

We will load at least 10 text files from the Python 3.14 documentation directory.

In [23]:
# Path to the text files directory
text_files_dir = "../UnstructureData/TextFiles/python-3.14-docs-text/"

# Get all .txt files in the directory
text_file_paths = glob.glob(os.path.join(text_files_dir, "*.txt"))

print(f"Found {len(text_file_paths)} text files")

# Load at least 10 text files
text_files_to_load = text_file_paths[:min(15, len(text_file_paths))]  # Load 15 files
print(f"Loading {len(text_files_to_load)} text files...")

# Initialize list to store all text chunks
text_chunks = []

# Load each text file
for file_path in text_files_to_load:
    try:
        loader = TextLoader(file_path, encoding='utf-8')
        # Load and split the file
        docs = loader.load()
        chunks = text_splitter.split_documents(docs)
        text_chunks.extend(chunks)
        print(f"  ✓ Loaded: {os.path.basename(file_path)} ({len(chunks)} chunks)")
    except Exception as e:
        print(f"  ✗ Error loading {os.path.basename(file_path)}: {e}")
        # Try with different encoding
        try:
            loader = TextLoader(file_path, encoding='latin-1')
            docs = loader.load()
            chunks = text_splitter.split_documents(docs)
            text_chunks.extend(chunks)
            print(f"  ✓ Loaded with latin-1: {os.path.basename(file_path)} ({len(chunks)} chunks)")
        except Exception as e2:
            print(f"  ✗ Failed with both encodings: {e2}")

print(f"\n✓ Total text chunks: {len(text_chunks)}")
print(f"\nSample text chunk (first 300 characters):")
if text_chunks:
    print(text_chunks[0].page_content[:300] + "...")

Found 8 text files
Loading 8 text files...
  ✓ Loaded: about.txt (3 chunks)
  ✓ Loaded: bugs.txt (9 chunks)
  ✓ Loaded: contents.txt (375 chunks)
  ✓ Loaded: copyright.txt (1 chunks)
  ✓ Loaded: glossary.txt (194 chunks)
  ✓ Loaded: improve-page-nojs.txt (1 chunks)
  ✓ Loaded: improve-page.txt (2 chunks)
  ✓ Loaded: license.txt (155 chunks)

✓ Total text chunks: 740

Sample text chunk (first 300 characters):
About this documentation
************************

Python's documentation is generated from reStructuredText sources
using Sphinx, a documentation generator originally created for Python
and now maintained as an independent project.

Development of the documentation and its toolchain is an entirely
...


## 7 - Load and Chunk HTML Files

We will load at least 10 HTML files from the pandas documentation directory.

In [24]:
# Path to the HTML files directory
html_files_dir = "../UnstructureData/HTMLFiles/pandasDoc"

# Get all .html files in the directory (including subdirectories)
html_file_paths = glob.glob(os.path.join(html_files_dir, "**/*.html"), recursive=True)

print(f"Found {len(html_file_paths)} HTML files")

# Load at least 10 HTML files
html_files_to_load = html_file_paths[:min(15, len(html_file_paths))]  # Load 15 files
print(f"Loading {len(html_files_to_load)} HTML files...")

# Initialize list to store all HTML chunks
html_chunks = []

# Load each HTML file
for file_path in html_files_to_load:
    try:
        loader = UnstructuredHTMLLoader(file_path)
        # Load and split the file
        docs = loader.load()
        chunks = text_splitter.split_documents(docs)
        html_chunks.extend(chunks)
        print(f"  ✓ Loaded: {os.path.basename(file_path)} ({len(chunks)} chunks)")
    except Exception as e:
        print(f"  ✗ Error loading {os.path.basename(file_path)}: {e}")

print(f"\n✓ Total HTML chunks: {len(html_chunks)}")
print(f"\nSample HTML chunk (first 300 characters):")
if html_chunks:
    print(html_chunks[0].page_content[:300] + "...")

Found 3850 HTML files
Loading 15 HTML files...
  ✓ Loaded: 10min.html (1 chunks)
  ✓ Loaded: advanced.html (1 chunks)
  ✓ Loaded: api.html (1 chunks)
  ✓ Loaded: basics.html (1 chunks)
  ✓ Loaded: categorical.html (1 chunks)
  ✓ Loaded: comparison_with_r.html (1 chunks)
  ✓ Loaded: comparison_with_sas.html (1 chunks)
  ✓ Loaded: comparison_with_sql.html (1 chunks)
  ✓ Loaded: comparison_with_stata.html (1 chunks)
  ✓ Loaded: computation.html (1 chunks)
  ✓ Loaded: contributing.html (1 chunks)
  ✓ Loaded: contributing_docstring.html (1 chunks)
  ✓ Loaded: cookbook.html (1 chunks)
  ✓ Loaded: developer.html (1 chunks)
  ✓ Loaded: dsintro.html (1 chunks)

✓ Total HTML chunks: 15

Sample HTML chunk (first 300 characters):
The page has been moved to 10 minutes to pandas...


## 8 - Combine All Document Chunks

Combine all chunks from PDF, Text, and HTML files into a single list for embedding.

In [25]:
# Combine all chunks
all_chunks = pdf_chunks + text_chunks + html_chunks

print(f"Total document chunks from all sources:")
print(f"  - PDF chunks: {len(pdf_chunks)}")
print(f"  - Text chunks: {len(text_chunks)}")
print(f"  - HTML chunks: {len(html_chunks)}")
print(f"  - TOTAL: {len(all_chunks)}")

# Verify we have documents from each type
assert len(pdf_chunks) > 0, "No PDF chunks loaded!"
assert len(text_chunks) >= 10, f"Expected at least 10 text file chunks, got {len(text_chunks)}"
assert len(html_chunks) >= 10, f"Expected at least 10 HTML file chunks, got {len(html_chunks)}"

print("\n✓ All document types successfully loaded and chunked!")

Total document chunks from all sources:
  - PDF chunks: 328
  - Text chunks: 740
  - HTML chunks: 15
  - TOTAL: 1083

✓ All document types successfully loaded and chunked!


## 9 - Create ChromaDB Vector Store

Now we will create embeddings for all document chunks and store them in ChromaDB for efficient retrieval.

**About Vector Databases:**
Vector databases like ChromaDB store document embeddings and enable fast similarity search. When you query the database:
1. Your query is converted to a vector
2. The database finds the most similar document vectors using cosine similarity
3. The top-k most similar documents are returned

This process is much faster than comparing your query to every document linearly.

In [26]:
# Create ChromaDB vector store from documents
# This will:
# 1. Generate embeddings for all chunks using our local embedding model
# 2. Store them in ChromaDB for efficient retrieval

print("Creating vector store with embeddings...")
print(f"This will process {len(all_chunks)} chunks...")
print("This may take a few minutes depending on the number of documents...")

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embedding_model,
    collection_name='Python_Documentation_RAG',
    persist_directory='./chroma_db'  # Save to disk for reuse
)

print("✓ Vector store created and persisted successfully!")
print(f"Collection name: Python_Documentation_RAG")
print(f"Number of documents indexed: {vectorstore._collection.count()}")

Creating vector store with embeddings...
This will process 1083 chunks...
This may take a few minutes depending on the number of documents...
✓ Vector store created and persisted successfully!
Collection name: Python_Documentation_RAG
Number of documents indexed: 2166


## 10 - Prepare the Retriever

Create a retriever that will fetch the most relevant document chunks for a given query.

**Retrieval Strategy:**
- **Search Type**: Similarity search using cosine similarity
- **k**: Number of top documents to retrieve (we use 5)

The retriever will convert your query to an embedding and find the 5 most similar document chunks.

In [27]:
# Create a retriever from the vector store
retriever = vectorstore.as_retriever(
    search_type='similarity',  # Use similarity search (cosine similarity)
    search_kwargs={'k': 5}     # Retrieve top 5 most relevant chunks
)

print("✓ Retriever configured successfully!")
print(f"Retrieval parameters:")
print(f"  - Search type: similarity")
print(f"  - Top-k documents: 5")

# Test the retriever
test_query = "What is a Python function?"
test_results = retriever.invoke(test_query)
print(f"\nTest retrieval for: '{test_query}'")
print(f"Retrieved {len(test_results)} documents")
print(f"\nFirst result preview (first 200 characters):")
print(test_results[0].page_content[:200] + "...")

✓ Retriever configured successfully!
Retrieval parameters:
  - Search type: similarity
  - Top-k documents: 5

Test retrieval for: 'What is a Python function?'
Retrieved 5 documents

First result preview (first 200 characters):
Chapter 6
Creating Functions in
Python
6.1 Introduction
A function is a block of code which only runs when it is called. You can pass
data, known as parameters, into a function. A function can return ...


## 11 - Design System Prompt for RAG

Design the system prompt that instructs the LLM on how to use the provided context to answer questions.

**Key Instructions:**
1. Only use the provided context to answer
2. Don't mention the context in the response
3. If the answer isn't in the context, say "I don't know"

In [28]:
# System message for the LLM
qna_system_message = """You are a helpful programming assistant whose task is to answer questions about Python programming using the provided context.

User input will have the context required to answer user questions.
This context will begin with the token: ###Context
The context contains relevant information from Python documentation, tutorials, and reference materials.

User questions will begin with the token: ###Question

Instructions:
- Answer questions ONLY using the information provided in the context
- Do not mention or reference the context in your final answer
- Provide clear, concise, and accurate answers
- If the answer is not found in the context, respond with "I don't know"
- When providing code examples, format them properly
- Be helpful and educational in your responses
"""

print("✓ System message configured")
print("\nSystem message preview:")
print(qna_system_message[:200] + "...")

✓ System message configured

System message preview:
You are a helpful programming assistant whose task is to answer questions about Python programming using the provided context.

User input will have the context required to answer user questions.
This...


## 12 - Design Query Template

Create a template for formatting the user query with retrieved context.

In [29]:
# Query template for combining context and question
qna_user_message_template = """###Context
Here are relevant documents that contain information to answer your question:

{context}

###Question
{question}

Please provide a clear and accurate answer based on the context above."""

print("✓ Query template configured")

✓ Query template configured


## 13 - Create RAG Function

Create the main RAG function that:
1. Takes a user question
2. Retrieves relevant document chunks
3. Formats the prompt with context
4. Generates a response using the local LLM

In [30]:
def RAG(user_question: str) -> str:
    """
    Retrieval-Augmented Generation function.
    
    Args:
        user_question: The question to answer using the RAG system
        
    Returns:
        Generated answer based on retrieved context
    """
    # Step 1: Retrieve relevant documents
    try:
        relevant_chunks = retriever.invoke(user_question)
        context_list = [doc.page_content for doc in relevant_chunks]
        context_for_query = "\n\n".join(context_list)
    except Exception as e:
        return f"Error during retrieval: {e}"
    
    # Step 2: Format the prompt
    formatted_prompt = f"""{qna_system_message}

{qna_user_message_template.format(context=context_for_query, question=user_question)}
"""
    
    # Step 3: Generate response using local LLM
    try:
        # Generate response
        response = llm_pipeline(
            formatted_prompt,
            max_new_tokens=512,
            temperature=0.1,
            top_p=0.95,
            repetition_penalty=1.2,
            do_sample=True
        )
        
        # Extract the generated text
        full_response = response[0]['generated_text']
        
        # Extract only the new generated text (remove the prompt)
        answer = full_response[len(formatted_prompt):].strip()
        
        return answer
        
    except Exception as e:
        return f"Error during generation: {e}"

print("✓ RAG function defined successfully!")

✓ RAG function defined successfully!


## 14 - Test RAG with Query Examples

Let's test the RAG system with several questions about Python programming.

### Example 1: Python Functions

In [31]:
query_1 = "What are Python functions and how do you define them?"

print(f"Question: {query_1}\n")
print("Answer:")
answer_1 = RAG(query_1)
print(answer_1)
print("\n" + "="*80 + "\n")

Question: What are Python functions and how do you define them?

Answer:
Based on the given context, here's an explanation of what Python functions are and how they're defined:

Functions in Python allow us to group related statements together so that we can reuse those statements later by calling the function again. They take inputs through arguments, perform some operations, and optionally produce outputs via returned values.

To define a simple function in Python, follow these steps:

1. Use the `def` keyword followed by the name of the function and parentheses containing any necessary parameter(s). For example:
   ```python
   def my_function(param):
       # Code inside the function goes here
   ```

2. Indicate where the body of the function starts - usually after two colons (`:`) at the end of its definition line. Inside the indented section, write down the logic of the function including variables, conditions, loops, calculations, etc., separated by semicolons if needed.

3. En

### Example 2: Python Data Structures

In [32]:
query_2 = "Explain the difference between lists and tuples in Python."

print(f"Question: {query_2}\n")
print("Answer:")
answer_2 = RAG(query_2)
print(answer_2)
print("\n" + "="*80 + "\n")

Question: Explain the difference between lists and tuples in Python.

Answer:
In Python, both lists and tuples are used to store collections of items. However, they differ significantly in terms of mutability and syntax usage.

Lists are mutable; this means their contents can be changed after creation. You define a list by enclosing elements within square brackets [], separated by commas. Here's an example:
```python
data_list = ['item', 'another item']
```
Tuples, on the other hand, are immutable once created. This implies that changes cannot alter its structure - adding new values or removing existing ones would result in creating a different object rather than modifying the original. They're defined similarly to lists except enclosed within parentheses () instead of square brackets []:
```python
data_tuple = ('element', 'other element')
```

Additionally, since tuples are less flexible due to immutability, certain operations might require converting them into lists if needed later. 

### Example 3: Pandas Operations

In [33]:
query_3 = "How do you read a CSV file using pandas and display the first few rows?"

print(f"Question: {query_3}\n")
print("Answer:")
answer_3 = RAG(query_3)
print(answer_3)
print("\n" + "="*80 + "\n")

Question: How do you read a CSV file using pandas and display the first few rows?

Answer:
To read a CSV file using pandas and display the first few rows, follow these steps:

```python
import pandas as pd

# Load the CSV file into a DataFrame
df = pd.read_csv('filename.csv')

# Display the first n rows (e.g., the first five)
print(df.head(n=5))
```

Replace `'filename.csv'` with the path to your actual CSV file. The `head()` function displays the initial parts of the dataframe by default showing the first ten lines unless otherwise specified (`n=X`). Adjust `n` if needed for more/less rows to be displayed. 

If there's no specific header row in the CSV file, consider specifying it like this:

```python
df = pd.read_csv('filename.csv', header=None) # No headers
```
Or specify column names manually if known:

```python
column_names=['col1','col2'] 
df = pd.read_csv('filename.csv',names=column_names)
```

Remember to replace 'filename.csv' with the correct filename/path depending upon wh

### Example 4: Python Classes

In [34]:
query_4 = "What are Python classes and how do you create them?"

print(f"Question: {query_4}\n")
print("Answer:")
answer_4 = RAG(query_4)
print(answer_4)
print("\n" + "="*80 + "\n")

Question: What are Python classes and how do you create them?

Answer:
In Python, a class serves as a blueprint for creating objects (a particular data structure). It defines a set of attributes which together make up the state of the object, along with functions defining operations on the state. To define a class named `Car`, you would write:

```python
class Car:
    model = 'Volvo'
```

With this syntax, you've created a basic class called `Car` where every instance has a common attribute `model`. This example demonstrates both the creation process and one typical feature within a class - setting initial values through class-level variables. You can further extend such a class by adding more member variables and methods according to specific needs. Creating instances of these classes allows you to instantiate unique objects while maintaining shared characteristics defined at the class level. ###Answer
A class in Python is a template for creating objects. It encapsulates data (attrib

## 15 - RAG Evaluation using LLM as Judge

We will now evaluate the quality of our RAG system using the LLM itself as a judge. This is known as "LLM-as-a-Judge" evaluation.

**Evaluation Metrics:**
1. **Groundedness/Faithfulness**: Is the answer derived from the provided context?
2. **Context Relevance**: Is the retrieved context relevant to the question?
3. **Answer Relevance**: Does the answer address the question asked?

For each metric, the LLM will provide:
- Step-by-step reasoning
- Detailed evaluation
- A score from 1-5

Let's use one of our previous questions for evaluation.

In [35]:
# Select a question for detailed evaluation
evaluation_question = query_2  # "Explain the difference between lists and tuples in Python."
evaluation_answer = answer_2

# Get the context that was used
relevant_chunks = retriever.invoke(evaluation_question)
context_list = [doc.page_content for doc in relevant_chunks]
evaluation_context = "\n\n".join(context_list)

print(f"Evaluation Question: {evaluation_question}")
print(f"\nEvaluation Answer Preview: {evaluation_answer[:200]}...")
print(f"\nContext chunks retrieved: {len(relevant_chunks)}")

Evaluation Question: Explain the difference between lists and tuples in Python.

Evaluation Answer Preview: In Python, both lists and tuples are used to store collections of items. However, they differ significantly in terms of mutability and syntax usage.

Lists are mutable; this means their contents can b...

Context chunks retrieved: 5


### Evaluation Metric 1: Groundedness/Faithfulness

**Groundedness** measures how much of the answer is derived from the provided context.

**Evaluation Criteria:**
- 1: The answer is not grounded in the context at all
- 2: The answer is grounded only to a limited extent
- 3: The answer is grounded to a good extent
- 4: The answer is mostly grounded in the context
- 5: The answer is completely grounded in the context

**Key Questions:**
- Does the answer introduce information not in the context?
- Are all factual claims supported by the context?
- Does the answer make unsupported assumptions?

In [36]:
# Template for groundedness evaluation
groundedness_evaluation_template = """###Question
{question}

###Context
{context}

###Answer
{answer}
"""

# System message for groundedness evaluation
groundedness_rater_system = """You are an expert evaluator assessing the groundedness of AI-generated answers.

You will be given:
- A question (marked with ###Question)
- Context used to answer the question (marked with ###Context)
- An AI-generated answer (marked with ###Answer)

Your task is to evaluate how well the answer is grounded in (derived from) the provided context.

Evaluation Criteria (1-5 scale):
1 - The answer is not grounded in the context at all
2 - The answer is grounded only to a limited extent
3 - The answer is grounded to a good extent
4 - The answer is mostly grounded in the context
5 - The answer is completely grounded in the context

Instructions:
1. List the steps needed to evaluate groundedness
2. Extract key information from the context
3. Compare the answer against the context to identify:
   - Information that comes from the context
   - Information that doesn't appear in the context
   - Any unsupported assumptions or external knowledge
4. Determine if all parts of the answer are derived solely from the context
5. Assign a score based on the evaluation criteria

Provide your evaluation in a clear, structured format."""

# Create the evaluation prompt
groundedness_prompt = f"""{groundedness_rater_system}

{groundedness_evaluation_template.format(
    question=evaluation_question,
    context=evaluation_context,
    answer=evaluation_answer
)}"""

print("Evaluating Groundedness/Faithfulness...")
print("This may take a moment...\n")

# Generate evaluation
groundedness_response = llm_pipeline(
    groundedness_prompt,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True
)

groundedness_evaluation = groundedness_response[0]['generated_text'][len(groundedness_prompt):].strip()

print("=" * 80)
print("GROUNDEDNESS/FAITHFULNESS EVALUATION")
print("=" * 80)
print(groundedness_evaluation)
print("=" * 80)

Evaluating Groundedness/Faithfulness...
This may take a moment...

GROUNDEDNESS/FAITHFULNESS EVALUATION
As seen above, attempting to modify a tuple directly leads to `TypeError`. Therefore, while using either structures depends largely upon specific needs regarding flexibility and performance characteristics, understanding these differences helps make informed decisions about choosing appropriate types during coding tasks involving arrays/sequences management in Python.

Based on the analysis:

#### Step-by-step Evaluation Process:

**Step 1:** Identify the necessary components required for evaluating groundedness including identifying relevant sections of text related to the topic being discussed ("lists", "tuples").

**Step 2:** Extract Key Information From Provided Text:
- **Key Points About Lists**: Mutable nature, definition via square bracket notation `[`, ability to add/remove elements (`append()`), examples shown through code snippets.
- **Key Points About Tuples**: Immutable n

### Evaluation Metric 2: Context Relevance

**Context Relevance** measures how relevant the retrieved context is to answering the question.

**Evaluation Criteria:**
- 1: The context is not relevant at all
- 2: The context is relevant only to a limited extent
- 3: The context is relevant to a good extent
- 4: The context is mostly relevant
- 5: The context is completely relevant

**Key Questions:**
- Does the context contain information needed to answer the question?
- Is there important information missing from the context?
- Is there irrelevant information in the context?

In [37]:
# Template for context relevance evaluation
context_relevance_evaluation_template = """###Question
{question}

###Context
{context}
"""

# System message for context relevance evaluation
context_relevance_rater_system = """You are an expert evaluator assessing the relevance of context used by an AI system.

You will be given:
- A question (marked with ###Question)
- Context used to answer the question (marked with ###Context)

Your task is to evaluate how relevant the context is to answering the question.

Evaluation Criteria (1-5 scale):
1 - The context is not relevant at all
2 - The context is relevant only to a limited extent
3 - The context is relevant to a good extent
4 - The context is mostly relevant
5 - The context is completely relevant

Context Relevance measures how well the provided context aligns with and supports answering the question.

Instructions:
1. List the steps needed to evaluate context relevance
2. Identify what information is needed to answer the question
3. Analyze the context to determine:
   - What relevant information is present
   - What necessary information is missing
   - What irrelevant information is included
4. Evaluate the extent to which the context supports answering the question
5. Assign a score based on the evaluation criteria

Provide your evaluation in a clear, structured format."""

# Create the evaluation prompt
context_relevance_prompt = f"""{context_relevance_rater_system}

{context_relevance_evaluation_template.format(
    question=evaluation_question,
    context=evaluation_context
)}"""

print("Evaluating Context Relevance...")
print("This may take a moment...\n")

# Generate evaluation
context_relevance_response = llm_pipeline(
    context_relevance_prompt,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True
)

context_relevance_evaluation = context_relevance_response[0]['generated_text'][len(context_relevance_prompt):].strip()

print("=" * 80)
print("CONTEXT RELEVANCE EVALUATION")
print("=" * 80)
print(context_relevance_evaluation)
print("=" * 80)

Evaluating Context Relevance...
This may take a moment...

CONTEXT RELEVANCE EVALUATION
To properly assess the relevance of the given context to the question about the differences between lists and tuples in Python, we'll follow these steps:

### Step-by-step Evaluation Process

#### **Step 1:** 
List the steps needed to evaluate context relevance.

**Steps:**
1. Determine key elements from the question that need addressing.
2. Examine each piece of information within the context against those elements.
3. Assess whether additional details would help clarify the topic further.
4. Score according to predefined criteria.

#### **Step 2:**
Identify what information is needed to answer the question.

**Needed Information:**
- Definition/characteristics of lists vs. tuples
- Differences specifically related to operations such as indexing, slicing, etc.
- Any specific examples involving both data structures

#### **Step 3:**

##### **What Relevant Information Is Present?**
The context contai

### Evaluation Metric 3: Answer Relevance

**Answer Relevance** measures how well the answer addresses the question asked.

**Evaluation Criteria:**
- 1: The answer is not relevant at all
- 2: The answer is relevant only to a limited extent
- 3: The answer is relevant to a good extent
- 4: The answer is mostly relevant
- 5: The answer is completely relevant

**Key Questions:**
- Does the answer directly address the question?
- Are all important aspects of the question covered?
- Is there unnecessary information in the answer?

In [38]:
# Template for answer relevance evaluation
answer_relevance_evaluation_template = """###Question
{question}

###Answer
{answer}
"""

# System message for answer relevance evaluation
answer_relevance_rater_system = """You are an expert evaluator assessing the relevance of AI-generated answers to questions.

You will be given:
- A question (marked with ###Question)
- An AI-generated answer (marked with ###Answer)

Your task is to evaluate how relevant the answer is to the question asked.

Evaluation Criteria (1-5 scale):
1 - The answer is not relevant at all
2 - The answer is relevant only to a limited extent
3 - The answer is relevant to a good extent
4 - The answer is mostly relevant
5 - The answer is completely relevant

Answer Relevance measures how well the answer addresses the main aspects of the question.

Instructions:
1. List the steps needed to evaluate answer relevance
2. Identify all important aspects of the question
3. Analyze the answer to determine:
   - Which aspects of the question are addressed
   - How thoroughly each aspect is covered
   - Whether any important aspects are missing
   - Whether there is unnecessary or off-topic information
4. Evaluate the extent to which the answer addresses all important aspects
5. Assign a score based on the evaluation criteria

Provide your evaluation in a clear, structured format."""

# Create the evaluation prompt
answer_relevance_prompt = f"""{answer_relevance_rater_system}

{answer_relevance_evaluation_template.format(
    question=evaluation_question,
    answer=evaluation_answer
)}"""

print("Evaluating Answer Relevance...")
print("This may take a moment...\n")

# Generate evaluation
answer_relevance_response = llm_pipeline(
    answer_relevance_prompt,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True
)

answer_relevance_evaluation = answer_relevance_response[0]['generated_text'][len(answer_relevance_prompt):].strip()

print("=" * 80)
print("ANSWER RELEVANCE EVALUATION")
print("=" * 80)
print(answer_relevance_evaluation)
print("=" * 80)

Evaluating Answer Relevance...
This may take a moment...

ANSWER RELEVANCE EVALUATION
As seen above, attempting to modify a tuple leads to a `TypeError`. Tuples also provide more efficient performance during lookups.
To convert a tuple back into a list use the following method:

```python
converted_to_list = list(data_tuple)
```

Based on these differences, choosing either one depends upon whether you need flexibility while changing content or prefer efficiency without modification opportunities.
### Evaluation Steps & Analysis

#### Step-by-step Evaluation Process

1. **Identify Key Aspects** – Break down the core components of the question.
   
    - Difference between Lists and Tuples
    - Mutability
    - Syntax Usage
    
2. **Analyze Answer Content**

    - Does the answer cover all key aspects?
    
    - Addressed Aspects:
        * Differences Between Lists and Tuples (Correctly identified but brief).
        * Mutability (Detailed explanation provided). 
        * Syntax Usa

## 16 - Evaluate Additional Examples

Let's evaluate our other query examples as well to get a comprehensive view of the RAG system's performance.

In [39]:
def evaluate_rag_response(question: str, answer: str, context: str) -> dict:
    """
    Evaluate a RAG response on all three metrics.
    
    Returns:
        Dictionary containing evaluations for all metrics
    """
    evaluations = {}
    
    # Groundedness evaluation
    groundedness_prompt = f"""{groundedness_rater_system}

{groundedness_evaluation_template.format(
    question=question,
    context=context,
    answer=answer
)}"""
    
    ground_resp = llm_pipeline(groundedness_prompt, max_new_tokens=512, temperature=0.1, do_sample=True)
    evaluations['groundedness'] = ground_resp[0]['generated_text'][len(groundedness_prompt):].strip()
    
    # Context relevance evaluation
    context_rel_prompt = f"""{context_relevance_rater_system}

{context_relevance_evaluation_template.format(
    question=question,
    context=context
)}"""
    
    ctx_rel_resp = llm_pipeline(context_rel_prompt, max_new_tokens=512, temperature=0.1, do_sample=True)
    evaluations['context_relevance'] = ctx_rel_resp[0]['generated_text'][len(context_rel_prompt):].strip()
    
    # Answer relevance evaluation
    ans_rel_prompt = f"""{answer_relevance_rater_system}

{answer_relevance_evaluation_template.format(
    question=question,
    answer=answer
)}"""
    
    ans_rel_resp = llm_pipeline(ans_rel_prompt, max_new_tokens=512, temperature=0.1, do_sample=True)
    evaluations['answer_relevance'] = ans_rel_resp[0]['generated_text'][len(ans_rel_prompt):].strip()
    
    return evaluations

print("✓ Evaluation function defined")

✓ Evaluation function defined


### Evaluate Query 1: Python Functions

In [40]:
# Get context for query 1
chunks_1 = retriever.invoke(query_1)
context_1 = "\n\n".join([doc.page_content for doc in chunks_1])

print(f"Evaluating Query 1: {query_1}")
print("This will take a few moments...\n")

eval_1 = evaluate_rag_response(query_1, answer_1, context_1)

print("=" * 80)
print(f"EVALUATION RESULTS FOR QUERY 1")
print("=" * 80)
print("\n--- GROUNDEDNESS ---")
print(eval_1['groundedness'])
print("\n--- CONTEXT RELEVANCE ---")
print(eval_1['context_relevance'])
print("\n--- ANSWER RELEVANCE ---")
print(eval_1['answer_relevance'])
print("=" * 80)

Evaluating Query 1: What are Python functions and how do you define them?
This will take a few moments...



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


EVALUATION RESULTS FOR QUERY 1

--- GROUNDEDNESS ---
``` To evaluate the groundedness of the AI-generated answer, I'll go step-by-step through each criterion:

### Step 1: Extract Key Information from the Context
Key points extracted from the context include:
- Definition of functions in Python
- Parameters passed to functions
- Return values produced by functions
- Built-in vs. User-defined functions
- Creating and using modules/libraries/packages
- Syntax for defining a function in Python

### Step 2: Identify Grounded Elements Against Answer Content
#### Comparison Table:
| **Context** | **AI Answer**                  |
|-------------|--------------------------------|
| Defining     | "Use the `def` keyword"         |
| Parameter    | "Take inputs through arguments"|
| Output       | "Produce output via returned values."|
| Types        | "User-defined vs. Builtin"      |

#### Analysis:
- **Defining**: Correctly mentions usage of `def`.
- **Parameter/Output**: Accurately describes 

### Evaluate Query 3: Pandas Operations

In [42]:
# Get context for query 3
chunks_3 = retriever.invoke(query_3)
context_3 = "\n\n".join([doc.page_content for doc in chunks_3])

print(f"Evaluating Query 3: {query_3}")
print("This will take a few moments...\n")

eval_3 = evaluate_rag_response(query_3, answer_3, context_3)

print("=" * 80)
print(f"EVALUATION RESULTS FOR QUERY 3")
print("=" * 80)
print("\n--- GROUNDEDNESS ---")
print(eval_3['groundedness'])
print("\n--- CONTEXT RELEVANCE ---")
print(eval_3['context_relevance'])
print("\n--- ANSWER RELEVANCE ---")
print(eval_3['answer_relevance'])
print("=" * 80)

Evaluating Query 3: How do you read a CSV file using pandas and display the first few rows?
This will take a few moments...

EVALUATION RESULTS FOR QUERY 3

--- GROUNDEDNESS ---
Let's go through the evaluation step-by-step according to the instructions.

#### Step 1: List the Steps Needed to Evaluate Groundedness
1. **Extract Key Information from the Context**
2. **Compare Answer Against Context**
    - Identify Information Derived From Context
    - Identify Unrelated Information
    - Check for Unsupported Assumptions or External Knowledge
3. **Determine If All Parts Are Derivable Solely From Context**

#### Step 2: Extract Key Information from the Context
**Key Points:** None relevant to answering the question about reading a CSV file using pandas and displaying its contents.

#### Step 3: Compare Answer Against Context
##### Identifying Relevant vs Irrelevant Information
- **Relevant Information**: There’s nothing directly related to the process described in the answer within the p

## 17 - Summary and Conclusions

### What We Accomplished

In this notebook, we successfully built a complete RAG system using local models:

1. **Local LLM**: Used Qwen2.5-3B-Instruct with 4-bit quantization for efficient inference
2. **Local Embeddings**: Used all-MiniLM-L6-v2 for generating document embeddings
3. **Multi-Format Documents**: Processed PDF, Text, and HTML files
4. **Vector Database**: Used ChromaDB for efficient similarity search
5. **RAG Pipeline**: Implemented retrieval-augmented generation
6. **Comprehensive Evaluation**: Evaluated outputs using LLM-as-judge on multiple metrics

### Key Advantages of Local Models

1. **Cost Efficiency**: No API costs for embeddings or inference
2. **Privacy**: All data processing happens locally
3. **Offline Capability**: Works without internet connection
4. **Customization**: Full control over models and parameters
5. **Scalability**: Can be deployed in production without API rate limits

### Performance Considerations

**Current Configuration:**
- Model: Qwen2.5-3B-Instruct (4-bit quantized)
- Embedding: all-MiniLM-L6-v2
- Chunk size: 512 characters
- Top-k retrieval: 5 documents

**Optimization Options:**
- Use GPU for faster inference (change `device='cpu'` to `device='cuda'`)
- Increase chunk_size for more context per chunk
- Adjust k value for more/fewer retrieved documents
- Use larger model (Qwen2.5-7B) for better quality
- Fine-tune embedding model for domain-specific tasks

### Next Steps

1. **Scale Up**: Process larger document collections
2. **Fine-tune**: Fine-tune models on domain-specific data
3. **Hybrid Search**: Combine dense and sparse retrieval
4. **Reranking**: Add a reranking step for better relevance
5. **Caching**: Implement caching for frequently asked questions
6. **UI/API**: Build a web interface or API for easier access

### Resources

- **Qwen2.5 Documentation**: https://huggingface.co/Qwen
- **Sentence Transformers**: https://www.sbert.net/
- **LangChain Documentation**: https://python.langchain.com/
- **ChromaDB Documentation**: https://docs.trychroma.com/

## Appendix: Troubleshooting

### Common Issues and Solutions

**Issue 1: Out of Memory Error**
- Solution: Use smaller model variant or reduce batch size
- Try: Qwen2.5-1.5B-Instruct instead of 3B

**Issue 2: Slow Inference**
- Solution: Use GPU if available
- Change: `device='cpu'` to `device='cuda'`
- Or increase quantization to 8-bit

**Issue 3: Poor Retrieval Quality**
- Solution: Adjust chunk size and overlap
- Try: chunk_size=1024, chunk_overlap=100
- Or increase k value for more retrieved documents

**Issue 4: File Loading Errors**
- Solution: Check file paths and encodings
- Try different encodings: utf-8, latin-1, cp1252

**Issue 5: Model Download Fails**
- Solution: Check internet connection
- Manually download from Hugging Face
- Or use cached models if available